In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
current_dir = Path().resolve()
sys.path.append(current_dir.parent.parent.as_posix())

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

import utils
from data_io import DataIO
from psth_utils import *
from psth_params import *

data_dir   = Path(r'/media/aleong/Audrey-experiments1') / 'Experiments' / 'dataset'
figure_dir = data_dir / 'Figures_summary'
os.makedirs(figure_dir, exist_ok=True)
print('Setup done.')


## Session configuration

In [ ]:
# ── Per-session configuration ─────────────────────────────────────────────────
# recording_nrs : [noblocker, acet, acet+lap4, washout] recording indices
# chirp_rec_nb  : 0-based index of the chirp recording used for cell typing
SESSION_CONFIG = {
    '260415_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260423_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260424_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260519_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260520_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
}

BLOCKER_ORDER = ['noblocker', 'acet', 'acet lap4', 'washout']
BLOCKER_LABEL = {
    'noblocker': 'No blocker',
    'acet':      'ACET',
    'acet lap4': 'ACET + LAP4',
    'washout':   'Washout',
}

CT_ORDER  = ['ON', 'OFF', 'ON-OFF', 'Unknown']
CT_COLORS = {'ON': '#E63946', 'OFF': '#457B9D', 'ON-OFF': '#9B59B6', 'Unknown': '#95A5A6'}

RESP_COLOR      = {'excitatory': '#E63946', 'inhibitory': '#457B9D'}
RESP_COLORS_CNT = {'excitatory': '#E63946', 'inhibitory': '#457B9D'}

SESSION_COLOR_LIST = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
SESSION_COLORS_MAP = dict(zip(SESSION_CONFIG.keys(), SESSION_COLOR_LIST))


## Load all sessions and compute PSTHs
For each session: load data, find preferred electrode per cell, compute PSTHs, accumulate into `df` (significant responses) and `zscore_store_all` (all conditions).  
Cell typing is also loaded per session when available.


In [ ]:
df_rows          = []
zscore_store_all = {}   # key: (cluster_id, blocker, prr, power)
typing_dfs       = {}   # key: session_id

for session_id, cfg in SESSION_CONFIG.items():
    print(f'\n{"="*60}')
    print(f'  {session_id}')
    print(f'{"="*60}')

    recording_nrs = cfg['recording_nrs']
    chirp_rec_nb  = cfg['chirp_rec_nb']

    # ── Load session ────────────────────────────────────────────
    data_io  = DataIO(data_dir)
    loadname = data_dir / f'{session_id}_cells.csv'
    data_io.load_session(session_id, load_pickle=True, load_waveforms=False)
    cells_df = pd.read_csv(loadname, header=[0, 1], index_col=0)

    # ── Select blocker-condition recordings ─────────────────────
    selected_rec_names = []
    for r in recording_nrs:
        for rec_id in data_io.recording_ids:
            if f'_{r:03d}_' in rec_id:
                selected_rec_names.append(rec_id)
                break
    if not selected_rec_names:
        print('  No matching recordings found — skipping.')
        continue
    print(f'  Recordings: {selected_rec_names}')

    # ── Preferred electrode per cell ────────────────────────────
    pref_ec_dict = {}
    for cluster_id in data_io.cluster_df.index.values:
        pref_ec, n_sig_pref_ec = None, None
        for ec in data_io.burst_df.electrode.unique():
            if pd.isna(ec):
                continue
            tids = data_io.burst_df.query(f'electrode == {float(ec)}').train_id.unique()
            n_sig = sum(
                1 for tid in tids
                if cells_df.loc[cluster_id, (tid, 'is_significant')] == True
            )
            if n_sig > 1 and (pref_ec is None or n_sig > n_sig_pref_ec):
                pref_ec, n_sig_pref_ec = ec, n_sig
        pref_ec_dict[cluster_id] = pref_ec

    n_responsive = sum(1 for v in pref_ec_dict.values() if v is not None)
    print(f'  Cells with preferred electrode: {n_responsive} / {len(pref_ec_dict)}')

    # ── PSTH computation ─────────────────────────────────────────
    cluster_ids = data_io.cluster_df.index.values

    for cluster_id in tqdm(cluster_ids, desc=f'  PSTHs'):
        ec = pref_ec_dict.get(cluster_id)
        if ec is None:
            continue

        for rec_name in selected_rec_names:
            d_select = data_io.burst_df.query(
                'electrode == @ec and recording_name == @rec_name'
            ).copy()
            if d_select.empty:
                continue

            blocker = d_select.Blocker.iloc[0]

            for prr in sorted(d_select.laser_pulse_repetition_rate.unique()):
                d_prr = d_select.query('laser_pulse_repetition_rate == @prr')

                for power in sorted(d_prr.laser_power.unique()):
                    d_pow = d_prr.query('laser_power == @power')
                    if d_pow.empty:
                        continue

                    tid      = d_pow.iloc[0].train_id
                    row0     = data_io.burst_df.query('train_id == @tid').iloc[0]
                    rec_id   = str(row0.rec_id)
                    stimtype = str(row0.stimtype)
                    bd       = float(row0.laser_burst_duration)

                    if stimtype in ('laser', 'padmd'):
                        burst_onsets = data_io.burst_df.query('train_id == @tid').laser_burst_onset.values
                    elif stimtype == 'dmd':
                        burst_onsets = data_io.burst_df.query('train_id == @tid').dmd_burst_onset.values
                    else:
                        continue

                    spiketrain = data_io.spiketimes[rec_id][cluster_id]

                    binned = []
                    for onset in burst_onsets:
                        idx = np.where(
                            (spiketrain >= onset + t_edges[0]) &
                            (spiketrain <  onset + t_edges[-1])
                        )[0]
                        counts, _ = np.histogram(spiketrain[idx] - onset, bins=t_edges)
                        binned.append(counts)

                    if not binned:
                        continue

                    binned        = np.vstack(binned)
                    spike_counts  = binned.sum(axis=0)
                    rate          = spike_counts / (len(binned) * (BIN_SIZE_MS / 1000))

                    baseline_mask = (t_centers >= -PRE_TIME_MS) & (t_centers < 0)
                    baseline_rate = rate[baseline_mask].mean()
                    if baseline_rate < MIN_BASELINE_HZ:
                        continue

                    smooth_sd   = get_adaptive_smooth_sd(baseline_rate)
                    rate_smooth = gaussian_filter1d(rate, smooth_sd)
                    mu          = rate_smooth[baseline_mask].mean()
                    sd          = rate_smooth[baseline_mask].std(ddof=1)

                    zscore      = (rate_smooth - mu) / sd if sd > 0 else np.zeros_like(rate_smooth)
                    zscore_norm = np.tanh(zscore / 3)

                    spike_counts_bl = spike_counts[baseline_mask]
                    results         = detect_latency(rate_smooth, mu, sd, spike_counts, spike_counts_bl)

                    # Mean post-stim rate in MEAN_WINDOW_MS
                    post_mask       = (t_centers >= MEAN_WINDOW_MS[0]) & (t_centers <= MEAN_WINDOW_MS[1])
                    mean_post_rate  = float(rate[post_mask].mean())

                    cond_key = (cluster_id, blocker, float(prr), float(power))
                    zscore_store_all[cond_key] = {
                        'session'       : session_id,
                        'zscore'        : zscore_norm,
                        'rate'          : rate_smooth,
                        'rate_raw'      : rate,
                        'mu'            : mu,
                        'sd'            : sd,
                        'latency'       : results['latency_ms'],
                        'response_type' : results['resp_type'],
                        'significant'   : results['significant'],
                        'bd'            : bd,
                        'mean_post_rate': mean_post_rate,
                    }

                    if not results['significant']:
                        continue

                    df_rows.append({
                        'session'       : session_id,
                        'cluster_id'    : cluster_id,
                        'blocker'       : blocker,
                        'prr'           : float(prr),
                        'power'         : float(power),
                        'response_type' : results['resp_type'],
                        'latency_ms'    : results['latency_ms'],
                        'baseline_rate' : mu,
                        'mean_post_rate': mean_post_rate,
                        'p_value'       : results['p_value'],
                    })

    # ── Load cell typing ─────────────────────────────────────────
    typing_directory = (
        Path(f'/media/aleong/Audrey-experiments1/Experiments/{session_id}/Analysis')
        / f'CellTyping_Analysis_rec_{chirp_rec_nb}'
    )
    typing_data_path = typing_directory / f'{session_id}_PA_ACET_1_cell_typing_data'
    try:
        typing_df = load_obj_as_df(typing_data_path)
        new_index = [f'uid_{session_id.split("_")[0]}_{i:03}' for i in typing_df.index]
        typing_df.index = new_index
        typing_dfs[session_id] = typing_df
        print(f'  Cell typing loaded: {len(typing_df)} cells')
    except Exception as e:
        print(f'  Cell typing not available: {e}')

df = pd.DataFrame(df_rows)
print(f'\nTotal significant responses across all sessions: {len(df)}')
if not df.empty:
    print(df.groupby(['blocker', 'response_type']).size().unstack(fill_value=0))


## Shared axis values and cell-type mapping

In [ ]:
# ── Collect all PRR / power values present across sessions ────────────────────
prr_vals   = sorted(df['prr'].unique())    if not df.empty else []
power_vals = sorted(df['power'].unique())  if not df.empty else []

df['prr_str']   = df['prr'].apply(lambda x: f'{int(x)} Hz')
df['power_str'] = df['power'].apply(lambda x: f'{int(x)} µW')
prr_order_str   = [f'{int(p)} Hz'  for p in prr_vals]
power_order_str = [f'{int(p)} µW'  for p in power_vals]

# ── Build combined typing lookup: cluster_id → cell_type / full_type_name ────
def _get_full_type(cell_type_value):
    """Return the full Baden type name (e.g. 'OFF alpha')."""
    if cell_type_value is None:
        return None
    if isinstance(cell_type_value, dict):
        return cell_type_value.get('name', None)
    if isinstance(cell_type_value, str):
        try:
            d = eval(cell_type_value)
            if isinstance(d, dict):
                return d.get('name', cell_type_value)
        except Exception:
            pass
        return cell_type_value
    return None

cid_to_ct       = {}
cid_to_fulltype = {}
for session_id, typing_df in typing_dfs.items():
    for cid in typing_df.index:
        btype = typing_df.loc[cid, 'baden_type']
        cid_to_ct[cid]       = extract_cell_type(btype) or 'Unknown'
        cid_to_fulltype[cid] = _get_full_type(btype)    or 'Unknown'

# ── Attach cell types to df ───────────────────────────────────────────────────
df['cell_type']  = df['cluster_id'].map(lambda c: cid_to_ct.get(c, 'Unknown'))
df['full_type']  = df['cluster_id'].map(lambda c: cid_to_fulltype.get(c, 'Unknown'))

HAS_TYPING = bool(typing_dfs)
print(f'Typing available: {HAS_TYPING}')
print(f'PRR values : {prr_vals}')
print(f'Power values: {power_vals}')


## Response latency — by PRR and laser power, per blocker condition
One figure per response type (excitatory / inhibitory).  
Rows = blocker conditions; left panel: x = PRR, right panel: x = laser power.  
Each point is one significant response; all sessions pooled.


In [ ]:
if df.empty:
    print('No significant responses — skipping latency plots.')
else:
    for resp_type in ['excitatory', 'inhibitory']:
        sub = df[df.response_type == resp_type].copy()
        if sub.empty:
            print(f'No {resp_type} responses.')
            continue

        color = RESP_COLOR[resp_type]
        n_blk = len(BLOCKER_ORDER)
        fig, axes = plt.subplots(
            n_blk, 2,
            figsize=(14, 3.5 * n_blk),
            constrained_layout=True,
        )

        for row_i, blocker in enumerate(BLOCKER_ORDER):
            sub_b = sub[sub.blocker == blocker]

            # Left: x = PRR, pooled across powers
            ax = axes[row_i, 0]
            sns.boxplot(data=sub_b, x='prr_str', y='latency_ms',
                        order=prr_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='prr_str', y='latency_ms',
                          order=prr_order_str, ax=ax,
                          hue='power_str', hue_order=power_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel(f'{BLOCKER_LABEL.get(blocker, blocker)}\nLatency (ms)', fontsize=9)
            ax.set_xlabel('PRR' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By PRR  (colour = power)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='Power', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

            # Right: x = power, pooled across PRRs
            ax = axes[row_i, 1]
            sns.boxplot(data=sub_b, x='power_str', y='latency_ms',
                        order=power_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='power_str', y='latency_ms',
                          order=power_order_str, ax=ax,
                          hue='prr_str', hue_order=prr_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel('')
            ax.set_xlabel('Laser power' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By laser power  (colour = PRR)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='PRR', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

        fig.suptitle(
            f'Latency — {resp_type.capitalize()} responses  '
            f'(n={len(sub)}, all sessions pooled)',
            fontsize=13, fontweight='bold',
        )
        fname = figure_dir / f'summary_latency_{resp_type}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()


## Firing rate — by PRR and laser power, per blocker condition
Mean firing rate in the post-stimulus window (`MEAN_WINDOW_MS`) for **significant** responses, pooled across all sessions.  
Layout mirrors the latency section.


In [ ]:
if df.empty:
    print('No significant responses — skipping firing rate plots.')
else:
    for resp_type in ['excitatory', 'inhibitory']:
        sub = df[df.response_type == resp_type].copy()
        if sub.empty:
            print(f'No {resp_type} responses.')
            continue

        color = RESP_COLOR[resp_type]
        n_blk = len(BLOCKER_ORDER)
        fig, axes = plt.subplots(
            n_blk, 2,
            figsize=(14, 3.5 * n_blk),
            constrained_layout=True,
        )

        for row_i, blocker in enumerate(BLOCKER_ORDER):
            sub_b = sub[sub.blocker == blocker]

            # Left: x = PRR
            ax = axes[row_i, 0]
            sns.boxplot(data=sub_b, x='prr_str', y='mean_post_rate',
                        order=prr_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='prr_str', y='mean_post_rate',
                          order=prr_order_str, ax=ax,
                          hue='power_str', hue_order=power_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel(
                f'{BLOCKER_LABEL.get(blocker, blocker)}\nMean firing rate (Hz)',
                fontsize=9,
            )
            ax.set_xlabel('PRR' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By PRR  (colour = power)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='Power', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

            # Right: x = power
            ax = axes[row_i, 1]
            sns.boxplot(data=sub_b, x='power_str', y='mean_post_rate',
                        order=power_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='power_str', y='mean_post_rate',
                          order=power_order_str, ax=ax,
                          hue='prr_str', hue_order=prr_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel('')
            ax.set_xlabel('Laser power' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By laser power  (colour = PRR)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='PRR', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

        fig.suptitle(
            f'Mean post-stim firing rate — {resp_type.capitalize()} responses  '
            f'(n={len(sub)}, all sessions pooled)',
            fontsize=13, fontweight='bold',
        )
        fname = figure_dir / f'summary_firing_rate_{resp_type}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()


## Response counts per cell type — by PRR and laser power, per blocker condition
Includes **all** cells (responders and non-responders) from `zscore_store_all`.  
One overall figure (majority vote across PRR/power) + one figure per blocker.


In [ ]:
if not HAS_TYPING:
    print('Cell typing data not available — skipping.')
else:
    # Build full table including non-responders
    _rows_all = []
    for (cid, blk, prr, pwr), d in zscore_store_all.items():
        resp = d['response_type'] if d['significant'] else 'none'
        _rows_all.append(dict(cluster_id=cid, blocker=blk,
                              prr=float(prr), power=float(pwr),
                              response_type=resp,
                              session=d['session']))
    df_all = pd.DataFrame(_rows_all)
    df_all['cell_type'] = df_all['cluster_id'].map(lambda c: cid_to_ct.get(c, 'Unknown'))
    df_all['resp_label'] = df_all['response_type'].map({
        'excitatory': 'Excitatory',
        'inhibitory': 'Inhibitory',
        'none':       'No response',
    })
    CT_PRESENT = [ct for ct in CT_ORDER if ct in df_all['cell_type'].unique()]
    RESP_LABELS_PLOT = ['Excitatory', 'Inhibitory', 'No response']
    RESP_COLORS_PLOT = {
        'Excitatory':  '#E63946',
        'Inhibitory':  '#457B9D',
        'No response': '#CCCCCC',
    }

    def _count_table_multi(sub):
        ct = (
            sub.groupby(['cell_type', 'resp_label'])
            .size()
            .unstack(fill_value=0)
            .reindex(index=CT_PRESENT, columns=RESP_LABELS_PLOT, fill_value=0)
        )
        ct  = ct.loc[ct.sum(axis=1) > 0]
        pct = ct.div(ct.sum(axis=1), axis=0) * 100
        return ct, pct

    def _plot_ct_panel(ax, counts, pct, title, show_ylabel=False):
        x       = np.arange(len(counts.index))
        n_resp  = len(RESP_LABELS_PLOT)
        width   = 0.22
        offsets = np.linspace(-(n_resp - 1) / 2, (n_resp - 1) / 2, n_resp) * width
        for resp, offset in zip(RESP_LABELS_PLOT, offsets):
            heights = [int(counts.loc[ct, resp]) if ct in counts.index else 0
                       for ct in counts.index]
            pcts    = [pct.loc[ct, resp]          if ct in pct.index    else 0.0
                       for ct in counts.index]
            bars = ax.bar(x + offset, heights, width,
                          color=RESP_COLORS_PLOT[resp], edgecolor='k',
                          linewidth=0.5, alpha=0.88, label=resp)
            for bar, h, p in zip(bars, heights, pcts):
                if h > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.05,
                            f'{h}\n({p:.0f}%)',
                            ha='center', va='bottom', fontsize=5.5,
                            fontweight='bold', color='#222')
        ax.set_xticks(x)
        ax.set_xticklabels(list(counts.index), fontsize=8)
        ax.set_xlabel('Cell type', fontsize=8)
        if show_ylabel:
            ax.set_ylabel('N cells', fontsize=8)
        ax.set_title(title, fontsize=8, fontweight='bold')
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)

    legend_handles = [
        mpatches.Patch(facecolor=RESP_COLORS_PLOT[r], edgecolor='k', label=r)
        for r in RESP_LABELS_PLOT
    ]

    # ── Overall figure (majority vote across PRR/power, all sessions) ─────────
    fig_ov, axes_ov = plt.subplots(
        len(BLOCKER_ORDER), 1,
        figsize=(7, 3.5 * len(BLOCKER_ORDER)),
        constrained_layout=True,
    )
    for ax_ov, blocker in zip(axes_ov, BLOCKER_ORDER):
        sub_b = df_all[df_all['blocker'] == blocker]
        if sub_b.empty:
            ax_ov.set_visible(False)
            continue
        majority = (
            sub_b.groupby(['cluster_id', 'cell_type'])['resp_label']
            .agg(lambda s: s.mode()[0])
            .reset_index()
        )
        counts_ov, pct_ov = _count_table_multi(majority)
        n_cells = sub_b['cluster_id'].nunique()
        _plot_ct_panel(ax_ov, counts_ov, pct_ov,
                       f'{BLOCKER_LABEL.get(blocker, blocker)}  (n={n_cells} cells, majority vote)',
                       show_ylabel=True)
    fig_ov.legend(handles=legend_handles, title='Response type',
                  loc='lower center', ncol=len(RESP_LABELS_PLOT),
                  bbox_to_anchor=(0.5, -0.02), fontsize=8, title_fontsize=9, framealpha=0.85)
    fig_ov.suptitle('Response type × cell type — overall (all sessions pooled)',
                    fontsize=12, fontweight='bold')
    fname = figure_dir / 'summary_cell_type_counts_overall.png'
    fig_ov.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()

    # ── Per-blocker figures: rows = PRR, cols = power ──────────────────────────
    prr_vals_all   = sorted(df_all['prr'].unique())
    power_vals_all = sorted(df_all['power'].unique())

    for blocker in BLOCKER_ORDER:
        sub_b = df_all[df_all['blocker'] == blocker]
        if sub_b.empty:
            continue

        fig, axes = plt.subplots(
            len(prr_vals_all), len(power_vals_all),
            figsize=(4.5 * len(power_vals_all), 3.5 * len(prr_vals_all)),
            constrained_layout=True,
            squeeze=False,
        )
        for row_i, prr in enumerate(prr_vals_all):
            for col_i, power in enumerate(power_vals_all):
                ax  = axes[row_i, col_i]
                sub = sub_b[(sub_b['prr'] == prr) & (sub_b['power'] == power)]
                if sub.empty:
                    ax.set_visible(False)
                    continue
                counts, pct = _count_table_multi(sub)
                _plot_ct_panel(ax, counts, pct,
                               f'PRR={int(prr/1000)} kHz | {int(power)} µW',
                               show_ylabel=(col_i == 0))

        fig.legend(handles=legend_handles, title='Response type',
                   loc='lower center', ncol=len(RESP_LABELS_PLOT),
                   bbox_to_anchor=(0.5, -0.04), fontsize=8, title_fontsize=9, framealpha=0.85)
        fig.suptitle(
            f'Response type × cell type — {BLOCKER_LABEL.get(blocker, blocker)}  (all sessions pooled)',
            fontsize=12, fontweight='bold',
        )
        fname = figure_dir / f'summary_cell_type_counts_{blocker.replace(" ", "_")}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()


## Which precise cell types respond to PA?
Using the **full Baden type name** (e.g. "OFF alpha", "ON sustained 1").  
Shows the number of cells with ≥1 significant PA response across all conditions, broken down by full type and by session.


In [ ]:
if not HAS_TYPING:
    print('Cell typing data not available — skipping.')
elif df.empty:
    print('No significant responses — skipping.')
else:
    # Per-cell summary: does this cell respond in any condition?
    df_full = df.copy()
    df_full['full_type'] = df_full['cluster_id'].map(lambda c: cid_to_fulltype.get(c, 'Unknown'))

    # Count unique responding cells per full Baden type (collapsed across conditions)
    responders_by_type = (
        df_full.groupby(['full_type', 'session'])['cluster_id']
        .nunique()
        .unstack(fill_value=0)
    )
    # Sort by total across sessions
    responders_by_type['_total'] = responders_by_type.sum(axis=1)
    responders_by_type = responders_by_type.sort_values('_total', ascending=False)
    responders_by_type = responders_by_type.drop(columns='_total')

    # ── Print summary table ───────────────────────────────────────────────────
    print('Unique responding cells per Baden type (any condition):')
    print(responders_by_type.to_string())

    # ── Bar chart ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(10, len(responders_by_type) * 0.6), 5),
                           constrained_layout=True)
    bottom = np.zeros(len(responders_by_type))
    for i, session_id in enumerate(sorted(df['session'].unique())):
        col = responders_by_type.get(session_id,
                                     pd.Series(0, index=responders_by_type.index))
        ax.bar(np.arange(len(responders_by_type)), col.values,
               bottom=bottom,
               color=SESSION_COLORS_MAP.get(session_id, f'C{i}'),
               label=session_id, edgecolor='k', linewidth=0.4, alpha=0.85)
        bottom += col.values

    ax.set_xticks(np.arange(len(responders_by_type)))
    ax.set_xticklabels(responders_by_type.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('N responding cells', fontsize=10)
    ax.set_xlabel('Baden cell type', fontsize=10)
    ax.set_title('Cells responding to PA — by precise Baden type  (any condition, all sessions)',
                 fontsize=11, fontweight='bold')
    ax.legend(title='Session', fontsize=8, title_fontsize=9, framealpha=0.85)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fname = figure_dir / 'summary_responding_cells_by_precise_type.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'\nSaved: {fname}')
    plt.show()

    # ── Also show excitatory vs inhibitory breakdown ───────────────────────────
    fig2, axes2 = plt.subplots(1, 2, figsize=(max(12, len(responders_by_type) * 1.2), 5),
                               constrained_layout=True)
    all_types = responders_by_type.index.tolist()

    for ax2, resp_type in zip(axes2, ['excitatory', 'inhibitory']):
        sub = df_full[df_full.response_type == resp_type]
        if sub.empty:
            ax2.set_visible(False)
            continue
        counts_rt = (
            sub.groupby(['full_type', 'session'])['cluster_id']
            .nunique()
            .unstack(fill_value=0)
            .reindex(index=all_types, fill_value=0)
        )
        bottom2 = np.zeros(len(all_types))
        for i, session_id in enumerate(sorted(df['session'].unique())):
            col = counts_rt.get(session_id, pd.Series(0, index=all_types))
            ax2.bar(np.arange(len(all_types)), col.values,
                    bottom=bottom2,
                    color=SESSION_COLORS_MAP.get(session_id, f'C{i}'),
                    label=session_id, edgecolor='k', linewidth=0.4, alpha=0.85)
            bottom2 += col.values

        ax2.set_xticks(np.arange(len(all_types)))
        ax2.set_xticklabels(all_types, rotation=45, ha='right', fontsize=8)
        ax2.set_ylabel('N responding cells', fontsize=9)
        ax2.set_title(f'{resp_type.capitalize()} responders', fontsize=10, fontweight='bold')
        ax2.spines[['top', 'right']].set_visible(False)
        ax2.grid(axis='y', alpha=0.3, linestyle='--')
        ax2.set_axisbelow(True)
        ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax2.legend(title='Session', fontsize=7, title_fontsize=8, framealpha=0.85)

    fig2.suptitle('PA-responding cells by precise Baden type and response polarity',
                  fontsize=11, fontweight='bold')
    fname2 = figure_dir / 'summary_responding_cells_by_type_exc_inh.png'
    fig2.savefig(fname2, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname2}')
    plt.show()


## Response evolution — cells responding under no-blocker AND ACET / ACET+LAP4
Filters to cells that have ≥1 significant response in:
- **No blocker** condition, AND
- **ACET** or **ACET + LAP4** condition.

For those cells, the response type in each blocker condition is shown as a stacked bar (majority vote across PRR/power).


In [ ]:
if df.empty:
    print('No significant responses — skipping.')
else:
    # ── Identify cells responding in noblocker ─────────────────────────────────
    nb_responders = set(
        df[df.blocker == 'noblocker']['cluster_id'].unique()
    )
    # Identify cells responding in acet OR acet lap4
    acet_responders = set(
        df[df.blocker.isin(['acet', 'acet lap4'])]['cluster_id'].unique()
    )
    # Intersection
    target_cells = nb_responders & acet_responders
    print(f'Cells responding in no-blocker: {len(nb_responders)}')
    print(f'Cells responding in ACET or ACET+LAP4: {len(acet_responders)}')
    print(f'Intersection (no-blocker AND ACET/LAP4): {len(target_cells)}')

    if not target_cells:
        print('No cells in intersection — skipping.')
    else:
        # Build full response table for target cells
        _rows_tc = []
        for (cid, blk, prr, pwr), d in zscore_store_all.items():
            if cid not in target_cells:
                continue
            resp = d['response_type'] if d['significant'] else 'none'
            _rows_tc.append(dict(
                cluster_id=cid, blocker=blk,
                prr=float(prr), power=float(pwr),
                response_type=resp,
                resp_label={'excitatory': 'Excitatory',
                            'inhibitory': 'Inhibitory',
                            'none':       'No response'}.get(resp, resp),
                session=d['session'],
            ))
        df_tc = pd.DataFrame(_rows_tc)
        df_tc['cell_type'] = df_tc['cluster_id'].map(lambda c: cid_to_ct.get(c, 'Unknown'))

        RESP_LABELS_PLOT = ['Excitatory', 'Inhibitory', 'No response']
        RESP_COLORS_PLOT = {
            'Excitatory':  '#E63946',
            'Inhibitory':  '#457B9D',
            'No response': '#CCCCCC',
        }

        # ── 1. Overall stacked bar: response type counts per blocker ───────────
        majority_per_cell = (
            df_tc.groupby(['cluster_id', 'blocker'])['resp_label']
            .agg(lambda s: s.mode()[0])
            .reset_index()
        )
        counts_per_blk = (
            majority_per_cell.groupby(['blocker', 'resp_label'])
            .size()
            .unstack(fill_value=0)
            .reindex(index=BLOCKER_ORDER, columns=RESP_LABELS_PLOT, fill_value=0)
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

        # Left: stacked bar by blocker
        ax = axes[0]
        bottom = np.zeros(len(BLOCKER_ORDER))
        for resp in RESP_LABELS_PLOT:
            vals = counts_per_blk[resp].values
            bars = ax.bar(
                np.arange(len(BLOCKER_ORDER)), vals,
                bottom=bottom,
                color=RESP_COLORS_PLOT[resp],
                label=resp, edgecolor='k', linewidth=0.5, alpha=0.88,
            )
            for bar, v, b in zip(bars, vals, bottom):
                if v > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, b + v / 2,
                            str(v), ha='center', va='center',
                            fontsize=9, fontweight='bold', color='k')
            bottom += vals

        ax.set_xticks(np.arange(len(BLOCKER_ORDER)))
        ax.set_xticklabels([BLOCKER_LABEL.get(b, b) for b in BLOCKER_ORDER],
                           rotation=20, ha='right', fontsize=9)
        ax.set_ylabel('N cells (majority vote)', fontsize=10)
        ax.set_title(
            f'Response type per blocker condition\n'
            f'(n={len(target_cells)} cells: no-blocker AND ACET/LAP4 responders)',
            fontsize=10, fontweight='bold',
        )
        ax.legend(fontsize=8, framealpha=0.85)
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

        # Right: per-cell heatmap (cluster × blocker → response type)
        ax2 = axes[1]
        # Encode response type as integer for colouring
        RESP_INT = {'Excitatory': 1, 'Inhibitory': -1, 'No response': 0}
        pivot = (
            majority_per_cell.pivot_table(
                index='cluster_id', columns='blocker',
                values='resp_label', aggfunc=lambda x: x.iloc[0],
            )
            .reindex(columns=BLOCKER_ORDER)
        )
        pivot_int = pivot.applymap(lambda x: RESP_INT.get(x, 0) if isinstance(x, str) else 0)
        cmap = plt.cm.get_cmap('RdBu', 3)
        im = ax2.imshow(pivot_int.values, aspect='auto', cmap=cmap,
                        vmin=-1.5, vmax=1.5, interpolation='nearest')
        ax2.set_xticks(np.arange(len(BLOCKER_ORDER)))
        ax2.set_xticklabels([BLOCKER_LABEL.get(b, b) for b in BLOCKER_ORDER],
                             rotation=25, ha='right', fontsize=8)
        ax2.set_yticks(np.arange(len(pivot_int)))
        ax2.set_yticklabels(pivot_int.index, fontsize=6)
        ax2.set_ylabel('Cell ID', fontsize=9)
        ax2.set_title('Per-cell response type across blockers', fontsize=10, fontweight='bold')
        cbar = plt.colorbar(im, ax=ax2, ticks=[-1, 0, 1])
        cbar.ax.set_yticklabels(['Inhibitory', 'No response', 'Excitatory'], fontsize=7)

        fname = figure_dir / 'summary_target_cells_blocker_comparison.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()

        # ── 2. Breakdown by cell type ──────────────────────────────────────────
        if HAS_TYPING:
            fig3, axes3 = plt.subplots(
                1, len(BLOCKER_ORDER),
                figsize=(4.5 * len(BLOCKER_ORDER), 5),
                constrained_layout=True,
            )
            CT_PRESENT_TC = [ct for ct in CT_ORDER if ct in df_tc['cell_type'].unique()]

            for ax3, blocker in zip(axes3, BLOCKER_ORDER):
                sub_b = df_tc[df_tc['blocker'] == blocker]
                if sub_b.empty:
                    ax3.set_visible(False)
                    continue
                majority_b = (
                    sub_b.groupby(['cluster_id', 'cell_type'])['resp_label']
                    .agg(lambda s: s.mode()[0])
                    .reset_index()
                )
                counts_b = (
                    majority_b.groupby(['cell_type', 'resp_label'])
                    .size()
                    .unstack(fill_value=0)
                    .reindex(index=CT_PRESENT_TC, columns=RESP_LABELS_PLOT, fill_value=0)
                )
                bottom3 = np.zeros(len(CT_PRESENT_TC))
                for resp in RESP_LABELS_PLOT:
                    vals3 = counts_b.get(resp, pd.Series(0, index=CT_PRESENT_TC)).values
                    ax3.bar(
                        np.arange(len(CT_PRESENT_TC)), vals3,
                        bottom=bottom3,
                        color=RESP_COLORS_PLOT[resp],
                        label=resp, edgecolor='k', linewidth=0.5, alpha=0.88,
                    )
                    bottom3 += vals3

                ax3.set_xticks(np.arange(len(CT_PRESENT_TC)))
                ax3.set_xticklabels(CT_PRESENT_TC, fontsize=9)
                ax3.set_xlabel('Cell type', fontsize=9)
                ax3.set_ylabel('N cells', fontsize=9)
                ax3.set_title(BLOCKER_LABEL.get(blocker, blocker),
                              fontsize=10, fontweight='bold')
                ax3.spines[['top', 'right']].set_visible(False)
                ax3.grid(axis='y', alpha=0.3)
                ax3.set_axisbelow(True)
                ax3.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
                if blocker == BLOCKER_ORDER[0]:
                    ax3.legend(fontsize=8, framealpha=0.85)

            fig3.suptitle(
                f'Response type per blocker × cell type\n'
                f'(n={len(target_cells)} cells: no-blocker AND ACET/LAP4 responders)',
                fontsize=11, fontweight='bold',
            )
            fname3 = figure_dir / 'summary_target_cells_by_celltype.png'
            fig3.savefig(fname3, dpi=150, bbox_inches='tight')
            print(f'Saved: {fname3}')
            plt.show()


## PSTHs — target cells (no-blocker AND ACET / ACET+LAP4 responders)
One figure per cell; layout: **rows = PRR**, **columns = laser power**.  
Each panel overlays the smoothed firing rate for all four blocker conditions (colour-coded).  
Solid lines = significant response; faded lines = non-significant.  
Dashed vertical marker = response latency (only for significant responses).  
Dotted horizontal lines = excitatory / inhibitory threshold from the **no-blocker** baseline.


In [ ]:
if df.empty or not target_cells:
    print('No target cells — skipping PSTH plots.')
else:
    BLOCKER_COLORS = {
        'noblocker': '#2196F3',   # blue
        'acet':      '#4CAF50',   # green
        'acet lap4': '#FF9800',   # orange
        'washout':   '#9E9E9E',   # gray
    }

    # Collect all PRR / power present across target cells
    _tc_keys  = [(cid, blk, prr, pwr) for (cid, blk, prr, pwr) in zscore_store_all
                 if cid in target_cells]
    all_prr   = sorted({k[2] for k in _tc_keys})
    all_power = sorted({k[3] for k in _tc_keys})

    out_dir_psth = figure_dir / 'target_cell_psths'
    os.makedirs(out_dir_psth, exist_ok=True)

    for cluster_id in tqdm(sorted(target_cells), desc='Plotting PSTHs'):
        cell_type = cid_to_ct.get(cluster_id, 'Unknown')
        full_type = cid_to_fulltype.get(cluster_id, 'Unknown')
        # Recover session from any zscore_store_all entry for this cell
        _ses = next(
            (d['session'] for (cid, _, _, _), d in zscore_store_all.items()
             if cid == cluster_id),
            '?'
        )

        # PRR / power values that actually have data for this cell
        cell_prr   = sorted({k[2] for k in _tc_keys if k[0] == cluster_id})
        cell_power = sorted({k[3] for k in _tc_keys if k[0] == cluster_id})
        if not cell_prr or not cell_power:
            continue

        fig, axes = plt.subplots(
            len(cell_prr), len(cell_power),
            figsize=(4.5 * len(cell_power), 3.5 * len(cell_prr)),
            squeeze=False,
            constrained_layout=True,
        )

        for row_i, prr in enumerate(cell_prr):
            for col_i, power in enumerate(cell_power):
                ax = axes[row_i, col_i]

                # Reference baseline from no-blocker condition
                nb_key = (cluster_id, 'noblocker', prr, power)
                mu_ref = sd_ref = bd_ms = None
                if nb_key in zscore_store_all:
                    _d = zscore_store_all[nb_key]
                    mu_ref = _d['mu']
                    sd_ref = _d['sd']
                    bd_ms  = _d['bd'] * 1000  # burst duration in ms

                has_any = False
                for blocker in BLOCKER_ORDER:
                    key = (cluster_id, blocker, prr, power)
                    if key not in zscore_store_all:
                        continue
                    d = zscore_store_all[key]
                    has_any = True

                    if bd_ms is None:
                        bd_ms = d['bd'] * 1000

                    color = BLOCKER_COLORS[blocker]
                    sig   = d['significant']
                    alpha = 1.0  if sig else 0.30
                    lw    = 1.8  if sig else 1.0
                    zord  = 3    if sig else 1

                    ax.plot(t_centers, d['rate'],
                            color=color, linewidth=lw, alpha=alpha, zorder=zord,
                            label=BLOCKER_LABEL.get(blocker, blocker))

                    if sig and not np.isnan(d['latency']):
                        ax.axvline(d['latency'], color=color,
                                   linewidth=1.2, linestyle='--',
                                   alpha=0.85, zorder=zord)

                if not has_any:
                    ax.set_visible(False)
                    continue

                # Threshold reference lines (from no-blocker baseline)
                if mu_ref is not None and sd_ref is not None:
                    ax.axhline(mu_ref,
                               color='#555', ls='--', lw=0.8, alpha=0.5, zorder=0)
                    ax.axhline(mu_ref + K_SD_EXCIT * sd_ref,
                               color=RESP_COLOR['excitatory'],
                               ls=':', lw=0.9, alpha=0.5, zorder=0)
                    ax.axhline(max(MIN_INHIB_THRESHOLD, mu_ref - K_SD_INHIB * sd_ref),
                               color=RESP_COLOR['inhibitory'],
                               ls=':', lw=0.9, alpha=0.5, zorder=0)

                # Stimulus window shading + onset line
                ax.axvline(0, color='k', ls='--', lw=0.9, alpha=0.55, zorder=0)
                if bd_ms is not None and bd_ms > 0:
                    ax.axvspan(0, bd_ms, color='gold', alpha=0.12, zorder=0)
                ax.axvspan(-PRE_TIME_MS, 0, color='lightcyan', alpha=0.12, zorder=0)

                ax.set_xlim(-PRE_TIME_MS, POST_TIME_MS)
                ax.set_title(f'PRR = {int(prr/1000)} kHz  |  {int(power)} µW',
                             fontsize=9, fontweight='bold')
                if row_i == len(cell_prr) - 1:
                    ax.set_xlabel('Time (ms)', fontsize=8)
                if col_i == 0:
                    ax.set_ylabel('Firing rate (Hz)', fontsize=8)
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(alpha=0.18)
                ax.tick_params(labelsize=7)

        # Legend: blocker colours
        _blockers_present = [
            b for b in BLOCKER_ORDER
            if any((cluster_id, b, prr, pwr) in zscore_store_all
                   for prr in cell_prr for pwr in cell_power)
        ]
        legend_lines = [
            Line2D([0], [0], color=BLOCKER_COLORS[b], lw=2,
                   label=BLOCKER_LABEL.get(b, b))
            for b in _blockers_present
        ] + [
            Line2D([0], [0], color='#555',                 lw=1.2, ls='--',
                   alpha=0.7, label='Baseline (no-blocker)'),
            Line2D([0], [0], color=RESP_COLOR['excitatory'], lw=1.0, ls=':',
                   alpha=0.7, label='Excit. threshold'),
            Line2D([0], [0], color=RESP_COLOR['inhibitory'], lw=1.0, ls=':',
                   alpha=0.7, label='Inhib. threshold'),
        ]
        fig.legend(handles=legend_lines,
                   loc='lower center', ncol=len(legend_lines),
                   bbox_to_anchor=(0.5, -0.06),
                   fontsize=8, framealpha=0.85)

        fig.suptitle(
            f'{cluster_id}  ·  {full_type}  ·  {_ses}',
            fontsize=11, fontweight='bold',
        )
        fname = out_dir_psth / f'psth_{cluster_id}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {len(target_cells)} PSTH figure(s) → {out_dir_psth}')
